In [ ]:
# FLAN RETRIVAL PLOTS
from glob import glob
import json
import pandas as pd

domain_groups = {
        'struct to text': ['web_nlg_en', 'dart', 'e2e_nlg', 'common_gen'],
        'commonsense': ['story_cloze', 'piqa', 'copa', 'hellaswag'],
        'sentiment': ['sst2', 'yelp_polarity_reviews', 'imdb_reviews', 'sentiment140'],
        'reading comp': ['multirc', 'squad_v2', 'squad_v1', 'openbookqa', 'bool_q', 'drop'],
        'closed_book QA': ['natural_questions', 'arc_easy', 'arc_challenge', 'trivia_qa'],
        'coreference': ['definite_pronoun_resolution', 'wsc'],
        'read.comp.w:commonsense': ['cosmos_qa', 'record'],
        'paraphrase': ['paws_wiki', 'glue_qqp', 'glue_mrpc', 'stsb'],
        'nli': ['cb', 'wnli', 'mnli_matched', 'anli_r3', 'anli_r2', 'anli_r1', 'mnli_mismatched', 'snli', 'qnli', 'rte'],
        'translation': ['wmt16_translate_tren', 'wmt16_translate_deen', 'wmt16_translate_ruen', 'wmt16_translate_fien',
                        'wmt16_translate_roen', 'wmt14_enfr', 'wmt16_translate_csen', 'para_crawl_enes'],
    }

def build_category_map():
    return {ds: domain for domain, datasets in domain_groups.items() for ds in datasets}

def process_entry(entry, meta, cat_map):
    entry['tag'] = entry['tag'].replace("_10templates", "")
    entry.update({k: meta[k] for k in ['k', 'fetch_k', 'threshold']})
    entry['k'] = int(entry['k'])
    entry['fetch_k'] = int(entry['fetch_k'])
    entry['threshold'] = float(entry['threshold'])

    loras = entry['retrieved_loras'].split("-")
    entry['retrieved_loras'] = loras
    entry['number_of_retrieved'] = len(loras)

    entry['loaded_correctly'] = int(entry['tag'] in loras)

    retrieved_domains = [cat_map.get(l) for l in loras if l in cat_map]
    domain = entry['domain']
    entry['loaded_correct_domain'] = int(domain in retrieved_domains)
    entry['all_loaded_correct_domain'] = int(all(domain == r for r in retrieved_domains))
    entry['perc_loaded'] = 1 / len(loras) if entry['tag'] in loras else 0

    return entry

paths = "../experiments/FLAN/retrival_experiments/*.json"
cat_map = build_category_map()
results = []

for file in glob(paths):
    with open(file, "r") as f:
        data = json.load(f)
        meta = data[0]
        results.extend([process_entry(entry, meta, cat_map) for entry in data[1:]])

results = pd.DataFrame(results)


In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
sns.set_theme(font_scale=1.6)

x_ticks = list(set(results['k']))
sns.set_style(style='ticks')
f = sns.relplot(x='k', y='loaded_correctly', hue='domain', style='domain',kind="line", data=results, markers=True, row="fetch_k", col="threshold", legend=True,)
sns.move_legend(f, loc='lower center', ncol=6, bbox_to_anchor=(.45, 1), title='FlanV2 Tasks')

plt.xticks(x_ticks)

plt.savefig("../figs/task_flan_big_task.pdf", bbox_inches = 'tight',
    pad_inches = 0)


In [ ]:

sns.set_theme(font_scale=1.6)

x_ticks = list(set(results['k']))
sns.set_style(style='ticks')
f = sns.relplot(x='k', y='loaded_correctly', hue='tag', style='tag',kind="line", data=results, markers=True, row="fetch_k", col="threshold", legend=True,)
sns.move_legend(f, loc='lower center', ncol=6, bbox_to_anchor=(.45, 1), title='FlanV2 Tasks')

plt.xticks(x_ticks)

plt.savefig("../figs/task_flan_big_task.pdf", bbox_inches = 'tight',
    pad_inches = 0)



In [ ]:
""" Connection between query domain and retrieved domains"""
import pandas as pd
import holoviews as hv

from holoviews import opts, dim
hv.extension('bokeh')

data_k20 = results[(results['k'] == 20) & (results['fetch_k'] == 200) & (results['threshold'] == 0.0)]

palette =  [
    "#4B8A3D",  # Forest Green
    "#C05C34",  # Rust
    "#F2C12E",  # Mustard Yellow
    "#B9D3C2",  # Light Sage Green
    "#C1A1B8",  # Lavender Gray
    "#6E4B3A",  # Mocha
    "#D1B79C",  # Light Taupe
    "#A6D8D3",  # Soft Aqua
    "#A67C52",  # Caramel
    "#B8B8B8"   # Light Gray
]

domains = list(set(cat_map.values()))
tags = [tag for d in domains for tag in domain_groups[d]]


def build_domain_mappings(domains, palette):
    map_domain_to_number = {d: idx for idx, d in enumerate(domains)}
    domain_colors = {d: palette[idx] for d, idx in map_domain_to_number.items()}
    return map_domain_to_number, domain_colors

def map_tags_to_domains(tags, tag_to_domain, map_domain_to_number):
    return {t: map_domain_to_number[tag_to_domain[t]] for t in tags}

def create_links(df, tag_map, domain_colors):
    links = [
        {
            'source': tag_map[row['tag']],
            'target': tag_map[t],
            'value': 1,
            'tag': row['tag'],
            'color': domain_colors[row['domain']],
            'domain': map_domain_to_number[row['domain']]
        }
        for _, row in df.iterrows()
        for t in row['retrieved_loras']
    ]
    return pd.DataFrame(links)

def normalize_column(df, col, new_col, out_min=0, out_max=4):
    min_val, max_val = df[col].min(), df[col].max()
    scale = lambda v: (v - min_val) / (max_val - min_val) * (out_max - out_min) + out_min
    df[new_col] = df[col].apply(scale)
    return df

def create_nodes(domains, domain_map, domain_colors):
    nodes = [{'name': d, 'idx': domain_map[d], 'color': domain_colors[d]} for d in domains]
    return hv.Dataset(pd.DataFrame(nodes), 'idx')

def build_chord_diagram(links, nodes):
    chord = hv.Chord((links, nodes)).select(value=(5, None)).opts(
        labels='name',
        width=400,
        height=400,
        edge_color='color',
        node_color='color'
    )
    return chord


map_domain_to_number, domain_colors = build_domain_mappings(domains, palette)
map_tag_to_domain_to_number = map_tags_to_domains(tags, cat_map, map_domain_to_number)

links_df = create_links(data_k20, map_tag_to_domain_to_number, domain_colors)
links_df = links_df.groupby(["source", "target", "tag", "color"]).sum().reset_index()
links_df = normalize_column(links_df, 'value', 'size')

nodes = create_nodes(domains, map_domain_to_number, domain_colors)
chord_diagram = build_chord_diagram(links_df, nodes)
chord_diagram

In [ ]:
map_tag_to_number = dict([(k, idx) for idx, k in enumerate(cat_map.keys())])

# Create links from a different DataFrame `df`
links_df = create_links(data_k20, map_tag_to_number, domain_colors)
links_df = links_df.groupby(["source", "target", "domain", "color"]).sum().reset_index()

links_df = normalize_column(links_df, 'value', 'size', out_min=0.1, out_max=2)

# Create and sort nodes
node_list = [
    {
        "name": t,
        "idx": map_tag_to_number[t],
        "domain": map_domain_to_number[cat_map[t]],
        "color": domain_colors[cat_map[t]]
    }
    for t in tags
]
nodes_df = pd.DataFrame(node_list).sort_values("domain")
nodes = hv.Dataset(nodes_df, 'idx')


# Build the chord diagram
chord = hv.Chord((links_df, nodes))

diagram = chord.opts(opts.Chord(
    edge_color="color",
    edge_line_width='size',
    labels='name',
    node_color='color',
    width=500,
    height=500,
)
)

chord

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import itertools
import numpy as np

cols = []
col_link =  [
    "#636EFA",
    "#EF553B",
    "#00CC96",
    "#AB63FA",
    "#FFA15A",
    "#19D3F3",
    "#FF6692",
    "#B6E880",
    "#FF97FF",
    "#FECB52",
]

tran = '0.6'
for i in col_link:
    h = i.lstrip('#')
    tup = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    res = ' '.join(str(val) for val in tup)
    res = res.replace(' ', ',')
    res = 'rgba(' + res + ',' + tran + ')'
    cols.append(res)


def name_to_num(name) -> int:
   return domains.index(name)

def build_matrix(domains):
    size_am = len(domains)
    matrix_domain = [[0] * size_am for _ in range(size_am)]

    for domain in domains:
        df_dom = data_k20[data_k20['domain'] == domain]
        ret_loras = df_dom['retrieved_loras']

        for rets in ret_loras:
            for target_tag in rets:
                target_domain = cat_map[target_tag]
                source = name_to_num(domain)
                target = name_to_num(target_domain)
                matrix_domain[source][target] = matrix_domain[source][target] + 1
    return matrix_domain

def sankey_no_loops(adj_matrix, node_labels, h, w):
    adj_matrix = np.array(adj_matrix)
    n = adj_matrix.shape[0]

    # If no labels provided, use string indices
    if node_labels is None:
        node_labels = [str(i) for i in range(n)]

    # Rename target nodes by appending suffix
    target_labels = [label + "_" for label in node_labels]

    # Combined nodes list: sources + renamed targets
    all_nodes = node_labels + target_labels

    source = []
    target = []
    value = []

    for i in range(n):
        for j in range(n):
            if adj_matrix[i, j] > 0:
                source.append(i)  # source index in node_labels
                target.append(n + j)  # target index shifted by n for renamed targets
                value.append(adj_matrix[i, j])

    fig = go.Figure(go.Sankey(
        node=dict(
            label=all_nodes,
            pad=10,
            thickness=20,
            line=dict(color="black", width=0.5),
            color =  [
               cols[i % len(cols)]
                for i in range(len(all_nodes))
            ],

        ),
        link=dict(
            source=source,
            target=target,
            value=value,
            color= [
                cols[i % len(cols)]
                for i in source
            ],
        ),
    ))

    fig.update_layout(
        font_size=14,
         width=w,
        height=h,
        )
    fig.show()

    fig.write_image("../figs/domain_sneaky.pdf", engine="kaleido")


matrix_domain = build_matrix(domains)
sankey_no_loops(matrix_domain, domains, 500, 700)

In [ ]:
## Finetune Results - comparison in table
import json
import pandas as pd

from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge
import numpy as np

## From https://github.com/StyxXuan/LoraRetriever/tree/main/

def calculate_bleu(references, candidates):
    scores = [sentence_bleu([ref.split()], cand.split()) for ref, cand in zip(references, candidates)]
    return np.round(np.mean(scores) * 100, 1) if scores else 0

# Function to calculate ROUGE score
def calculate_rouge(references, candidates):
    rouge = Rouge()
    scores = rouge.get_scores(candidates, references, avg=True)
    rouge_1 = np.round(scores['rouge-1']['f'] * 100, 1)
    rouge_2 = np.round(scores['rouge-2']['f'] * 100, 1)
    rouge_l = np.round(scores['rouge-l']['f'] * 100, 1)
    return rouge_1, rouge_2, rouge_l

# Function to calculate Exact Match score
def calculate_em(references, candidates):
    references = [ref.split("\n\n")[0] for ref in references]
    em_scores = [1 if cal_correct(ref, cand) else 0 for ref, cand in zip(references, candidates)]
    return np.round(np.mean(em_scores) * 100, 1) if em_scores else 0

def cal_correct(generated_answer, expected_answer):
    is_correct = generated_answer.strip().lower().replace(".", "") == expected_answer.strip().lower().replace(".", "")
    return is_correct

with open("../experiments/FLAN/inference/d6b8d870c6de6ffa5c5b1889317d5a88.json", "r") as fp:
    data = json.load(fp)

data = pd.DataFrame(data[1:])


In [ ]:
# we use the same scoring system as LoRARetriver
from collections import defaultdict
organized_data = defaultdict(lambda: defaultdict(list))

for index,entry in data.iterrows():
    domain = entry['domain']
    task = entry['tag']
    organized_data[domain][task].append(entry)

domain_specific_metrics = []
for domain, tasks_data in organized_data.items():
    for task, entries in tasks_data.items():
        metric = entries[0]['metric']
        references = [entry['targets'] for entry in entries]
        candidates = [entry['predictions'][0] for entry in entries]
        task = task.replace("_10templates", "")
        if metric == 'bleu':
            score = calculate_bleu(references, candidates)
            domain_specific_metrics.append({"domain":domain,"tasks":task, "score":score})
        elif metric == 'rouge':
            score = calculate_rouge(references, candidates)
            domain_specific_metrics.append({"domain":domain,"tasks":task, "score":score})
        elif metric == 'em':
            score = calculate_em(references, candidates)
            domain_specific_metrics.append({"domain":domain,"tasks":task, "score":score})

scores = pd.DataFrame(domain_specific_metrics)
scores